# OWID Energy — Part 1: Exploratory Data Analysis

This notebook preprocesses energy source/generation data from Our World in Data to be used in a Tableau dashboard. This exploratory data analysis reviews the data, confirms the primary key, identifies missing values, and validates the key metrics to build a dashboard-ready dataset.

The purpose of the dashboard is to track how each country's energy mix has shifted between fossil and renewable sources over time. The dashboard is intended as an exploratory view of country-level patterns in the global energy transition.

*Note: The dashboard targets the 2000–2024 window; 2025 is excluded due to incomplete reporting (only ~40% of countries reporting).*

**Pipeline Position:** Notebook 1 of 1 — Exploratory Analysis
- 01_exploratory_analysis.ipynb ← this notebook

**Objective:** Produce a Tableau-ready subset of OWID energy data (country-level rows only) that contains full coverage on the five anchor share metrics.

**Technical Approach:**
1. Subset to 31 dashboard-relevant columns and 25 year date range (2000–2024)
2. Confirm the primary key (country, year) has no nulls or duplicates
3. Evaluate missing values
4. Validate that renewables + fossil shares sum to ~100
5. Filter to countries with full-coverage and export the subset


**Inputs:**
- `data/owid-energy-data.csv` — country-level energy data (130 cols × 23,377 rows)
- `data/owid-energy-codebook.csv` — variable definitions, units, source notes (5 cols × 130 rows)

**Outputs:**
- `data/processed/owid_energy_dashboard_subset.csv` — filtered subset for Tableau (79 countries × 25 years × 31 cols = 1,975 rows)

**Author:** K Flowers  
**Date:** January 2026

**Table of Contents**
1. Configure Environment
2. Load and Subset Data
3. Confirm Primary Key
4. Assess Missing Data
5. Validate Key Metrics
6. Final Selection and Export
7. Conclusion

## 1. Configure Environment

In [1]:
# Standard library
from pathlib import Path
import warnings

# Core data libraries
import pandas as pd
import numpy as np

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Project paths
PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"

# Create processed directory if needed
(DATA_DIR / "processed").mkdir(parents=True, exist_ok=True)

print("Libraries imported successfully")

Libraries imported successfully


## 2. Load and Subset Data

- Load energy and codebook CSVs
- Review structure (shape, dtypes, head)
- Select dashboard-relevant columns
- Filter to country-level rows, 2000–2024
- **Output:** filtered dataframe with the rows and columns needed for analysis

### 2.1 Load Datasets

In [2]:
# Load the OWID energy dataset and codebook from CSV
energy_path = DATA_DIR / "owid-energy-data.csv"
codebook_path = DATA_DIR / "owid-energy-codebook.csv"
energy_df = pd.read_csv(energy_path)
codebook_df = pd.read_csv(codebook_path)

print(f"Energy dataset loaded: {energy_path.name}")
print(f"\nCodebook loaded: {codebook_path.name}")

Energy dataset loaded: owid-energy-data.csv

Codebook loaded: owid-energy-codebook.csv


## 2.2 Enrich with continent assignments

Add a `continent` column derived from `iso_code` using `pycountry-convert` to be used in dashboard.

In [3]:
# Add a continent column to energy_df by mapping each row's iso_code.
# Why:  enables continent grouping in Tableau without a second script.

import pycountry_convert as pc


def iso_to_continent(iso3: str) -> str:
    """Map ISO-3 to continent. 'Global' for World, 'Unknown' for unmapped codes."""
    if pd.isna(iso3) or iso3 == '':
        return 'Global'
    try:
        iso2 = pc.country_alpha3_to_country_alpha2(iso3)
        continent_code = pc.country_alpha2_to_continent_code(iso2)
        return pc.convert_continent_code_to_continent_name(continent_code)
    except KeyError:
        return 'Unknown'


energy_df['continent'] = energy_df['iso_code'].apply(iso_to_continent)

# Sanity check: distribution and any unmapped codes
print(energy_df['continent'].value_counts(dropna=False))
unknown = sorted(energy_df.loc[energy_df['continent'] == 'Unknown', 'iso_code'].dropna().unique())
print(f"\nUnmapped iso_codes: {unknown if unknown else 'none'}")

continent
Global           6112
Asia             4574
Africa           4351
Europe           3516
North America    2270
South America    1372
Oceania          1012
Unknown           170
Name: count, dtype: int64

Unmapped iso_codes: ['ANT', 'ATA', 'ESH', 'TLS']


### 2.3 Review Data Structure

In [4]:
print(f"Energy dataset shape: {energy_df.shape[0]:,} rows × {energy_df.shape[1]} columns")
print(f"Codebook shape:       {codebook_df.shape[0]:,} rows × {codebook_df.shape[1]} columns\n")
energy_df.info()

Energy dataset shape: 23,377 rows × 131 columns
Codebook shape:       130 rows × 5 columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23377 entries, 0 to 23376
Columns: 131 entries, country to continent
dtypes: float64(127), int64(1), object(3)
memory usage: 23.4+ MB


In [5]:
# Display the first few rows of the energy dataset amd verify data loaded correctly
energy_df.head()

,country,year,iso_code,population,gdp,biofuel_cons_change_pct,biofuel_cons_change_twh,biofuel_cons_per_capita,biofuel_consumption,biofuel_elec_per_capita,biofuel_electricity,biofuel_share_elec,biofuel_share_energy,carbon_intensity_elec,coal_cons_change_pct,coal_cons_change_twh,coal_cons_per_capita,coal_consumption,coal_elec_per_capita,coal_electricity,coal_prod_change_pct,coal_prod_change_twh,coal_prod_per_capita,coal_production,coal_share_elec,coal_share_energy,electricity_demand,electricity_demand_per_capita,electricity_generation,electricity_share_energy,energy_cons_change_pct,energy_cons_change_twh,energy_per_capita,energy_per_gdp,fossil_cons_change_pct,fossil_cons_change_twh,fossil_elec_per_capita,fossil_electricity,fossil_energy_per_capita,fossil_fuel_consumption,fossil_share_elec,fossil_share_energy,gas_cons_change_pct,gas_cons_change_twh,gas_consumption,gas_elec_per_capita,gas_electricity,gas_energy_per_capita,gas_prod_change_pct,gas_prod_change_twh,gas_prod_per_capita,gas_production,gas_share_elec,gas_share_energy,greenhouse_gas_emissions,hydro_cons_change_pct,hydro_cons_change_twh,hydro_consumption,hydro_elec_per_capita,hydro_electricity,hydro_energy_per_capita,hydro_share_elec,hydro_share_energy,low_carbon_cons_change_pct,low_carbon_cons_change_twh,low_carbon_consumption,low_carbon_elec_per_capita,low_carbon_electricity,low_carbon_energy_per_capita,low_carbon_share_elec,low_carbon_share_energy,net_elec_imports,net_elec_imports_share_demand,nuclear_cons_change_pct,nuclear_cons_change_twh,nuclear_consumption,nuclear_elec_per_capita,nuclear_electricity,nuclear_energy_per_capita,nuclear_share_elec,nuclear_share_energy,oil_cons_change_pct,oil_cons_change_twh,oil_consumption,oil_elec_per_capita,oil_electricity,oil_energy_per_capita,oil_prod_change_pct,oil_prod_change_twh,oil_prod_per_capita,oil_production,oil_share_elec,oil_share_energy,other_renewable_consumption,other_renewable_electricity,other_renewable_exc_biofuel_electricity,other_renewables_cons_change_pct,other_renewables_cons_change_twh,other_renewables_elec_per_capita,other_renewables_elec_per_capita_exc_biofuel,other_renewables_energy_per_capita,other_renewables_share_elec,other_renewables_share_elec_exc_biofuel,other_renewables_share_energy,per_capita_electricity,primary_energy_consumption,renewables_cons_change_pct,renewables_cons_change_twh,renewables_consumption,renewables_elec_per_capita,renewables_electricity,renewables_energy_per_capita,renewables_share_elec,renewables_share_energy,solar_cons_change_pct,solar_cons_change_twh,solar_consumption,solar_elec_per_capita,solar_electricity,solar_energy_per_capita,solar_share_elec,solar_share_energy,wind_cons_change_pct,wind_cons_change_twh,wind_consumption,wind_elec_per_capita,wind_electricity,wind_energy_per_capita,wind_share_elec,wind_share_energy,continent
0,ASEAN (Ember),2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.87,1.55,NaN,572.58,NaN,NaN,NaN,NaN,NaN,76.03,NaN,NaN,NaN,NaN,20.07,NaN,378.76,NaN,378.76,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,305.53,NaN,NaN,80.67,NaN,NaN,NaN,NaN,NaN,164.26,NaN,NaN,NaN,NaN,NaN,43.37,NaN,216.87,NaN,NaN,NaN,NaN,50.43,NaN,13.31,NaN,NaN,NaN,NaN,NaN,73.23,NaN,19.33,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,NaN,0.00,NaN,NaN,NaN,NaN,NaN,65.24,NaN,NaN,NaN,NaN,NaN,17.23,NaN,NaN,22.80,16.93,NaN,NaN,NaN,NaN,NaN,6.02,4.47,NaN,NaN,NaN,NaN,NaN,NaN,NaN,73.23,NaN,19.33,NaN,NaN,NaN,NaN,NaN,0.00,NaN,0.00,NaN,NaN,NaN,NaN,NaN,0.00,NaN,0.00,NaN,Global
1,ASEAN (Ember),2001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.46,1.59,NaN,570.27,NaN,NaN,NaN,NaN,NaN,86.26,NaN,NaN,NaN,NaN,21.29,NaN,405.09,NaN,405.09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,327.90,NaN,NaN,80.94,NaN,NaN,NaN,NaN,NaN,190.41,NaN,NaN,NaN,NaN,NaN,47.00,NaN,231.01,NaN,NaN,NaN,NaN,54.33,NaN,13.41,NaN,NaN,NaN,NaN,NaN,77.19,NaN,19.05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,NaN,0.00,NaN,NaN,NaN,NaN,NaN,51.23,NaN,NaN,NaN,NaN,NaN,12.65,NaN,NaN,22.86,16.40,NaN,NaN,NaN,NaN,NaN,5.64,4.05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,77.19,NaN,19.05,NaN,NaN,NaN,NaN,NaN,0.00,NaN,0.00,NaN,NaN,NaN,NaN,NaN,0.00,NaN

In [6]:
# reveiw the codebook to verify it loaded correctly and understand the variable descriptions
codebook_df.head()

,column,title,description,unit,source
0,country,Country,Geographic location.,NaN,Our World in Data - Regions (2025)
1,year,Year,Year of observation.,NaN,Our World in Data - Regions (2025)
2,iso_code,ISO code,ISO 3166-1 alpha-3 three-letter country codes.,NaN,International Organization for Standardization...
3,population,Population,"Population by country, available from 10,000 B...",people,Population based on various sources (2024) [ht...
4,gdp,Gross domestic product (GDP),Total economic output of a country or region p...,international-$ in 2011 prices ($),Bolt and van Zanden – Maddison Project Databas...


### 2.4 Select Columns

In [7]:
# Column filter: dashboard-relevant variables, chosen from codebook + dashboard design.
# The actual subset is applied alongside the row filter in 2.5.

DASHBOARD_COLUMNS = [
    # Identifiers
    'country', 'year', 'iso_code', 'continent',

    # Category totals — renewable / fossil / low-carbon — both views
    'renewables_share_elec', 'renewables_share_energy',
    'fossil_share_elec', 'fossil_share_energy',
    'low_carbon_share_elec', 'low_carbon_share_energy',

    # Low-carbon breakdown (renewables + nuclear) — electricity
    'solar_share_elec', 'wind_share_elec', 'hydro_share_elec',
    'nuclear_share_elec', 'biofuel_share_elec',

    # Low-carbon breakdown (renewables + nuclear) — total primary energy
    'solar_share_energy', 'wind_share_energy', 'hydro_share_energy',
    'nuclear_share_energy', 'biofuel_share_energy',

    # Fossil breakdown — electricity
    'coal_share_elec', 'gas_share_elec', 'oil_share_elec',

    # Fossil breakdown — total primary energy
    'coal_share_energy', 'gas_share_energy', 'oil_share_energy',

    # Context (sizing, per-capita / per-GDP framing, tooltips)
    'population', 'gdp', 'primary_energy_consumption',
    'electricity_generation', 'electricity_demand',

    # Optional decarbonization angle
    'carbon_intensity_elec',
]

print(f"Selected {len(DASHBOARD_COLUMNS)} of {energy_df.shape[1]} columns for the dashboard.")

Selected 32 of 131 columns for the dashboard.


### 2.5 Select Date Range

In [8]:
# Final filter: country-level rows (iso_code non-null), 2000–2024, dashboard columns only.
filtered_df = energy_df[
    (energy_df['iso_code'].notna() | (energy_df['country'] == 'World')) &
    (energy_df['year'] >= 2000) &
    (energy_df['year'] <= 2024)
][DASHBOARD_COLUMNS].copy()

print(f"Filtered dataset: {filtered_df.shape[0]:,} rows × {filtered_df.shape[1]} columns")

Filtered dataset: 5,484 rows × 32 columns


In [9]:
# Review country count by year to verify coverage (years without sufficient country coverage may need to be excluded from the dashboard)
print(f"Date range:        {filtered_df['year'].min()}–{filtered_df['year'].max()}")
print(f"Unique countries:  {filtered_df['country'].nunique()}")
print()
print("Countries per year")
print("-" * 20)
print(filtered_df.groupby('year')['country'].nunique().to_string())

Date range:        2000–2024
Unique countries:  221

Countries per year
--------------------
year
2000    218
2001    218
2002    218
2003    219
2004    219
2005    220
2006    220
2007    220
2008    220
2009    220
2010    220
2011    220
2012    221
2013    221
2014    221
2015    221
2016    221
2017    221
2018    221
2019    221
2020    221
2021    221
2022    221
2023    221
2024    200


**Dashboard Note:** Country count per year shows 2025 has only 90 of 220 countries (~60% missing).  Dataset is filtered to 2000–2024 to ensure adequate coverage. This may be early-release data and not yet available.

- **2000–2023:** consistently covered (217–220 of 220 countries each year)
- **2024:** 199 of 220 — ~10% of countries missing
- **2025:** 90 of 220 — ~60% of countries missing

## 3. Confirm Primary Key

- Verify country identity — every row is either a country with a non-null `iso_code` or the `World` aggregate
- Verify uniqueness — exactly one row per `(country, year)`

### 3.1 Verify Country Identity (iso_code)

In [10]:
# What: confirm the row filter kept only countries (with iso_code) plus the World aggregate.
# Why: every remaining entity should be either a single country or the World total — no
#      regional aggregates or income groups, which would double-count if charted alongside
#      countries.

print('Unique entities:', filtered_df['country'].nunique())

# Every row should have iso_code OR be the World aggregate
non_world = filtered_df[filtered_df['country'] != 'World']
print('All non-World rows have iso_code:', non_world['iso_code'].notna().all())
print('World rows present:', (filtered_df['country'] == 'World').sum(), '(expected 25)')

print('\nFirst 20 countries (alphabetical):')
print(sorted(filtered_df['country'].unique())[:20])

Unique entities: 221
All non-World rows have iso_code: True
World rows present: 25 (expected 25)

First 20 countries (alphabetical):
['Afghanistan', 'Albania', 'Algeria', 'American Samoa', 'Angola', 'Antarctica', 'Antigua and Barbuda', 'Argentina', 'Armenia', 'Aruba', 'Australia', 'Austria', 'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barbados', 'Belarus', 'Belgium', 'Belize']


**Dashboard Note:** OWID embeds aggregates (`World`, `Europe`, income groups) alongside countries in the same `country` column. This dataset keeps **only individual countries plus the global `World` aggregate** — regional aggregates (`Europe`, `Asia`, income groups, etc.) are dropped via the `iso_code` filter. In Tableau, treat `World` as a separate series (not a sum of countries) to avoid double-counting against the country-level rows.

### 3.2 Verify Uniqueness (country, year)

In [11]:
# Why: country-year is the natural primary key. Duplicates would distort time-series
#      charts and inflate any aggregation in Tableau.

dupes = filtered_df.duplicated(subset=['country', 'year']).sum()
print(f"Duplicate country-year rows: {dupes}")

Duplicate country-year rows: 0


## 4. Assess Missing Data

- Build a per-column inventory: dtype + non-null count + percent missing + codebook description
- Interpret the pattern of missingness (systematic vs. random)

In [12]:
# What: every column in the filtered dataset with its dtype, non-null count, percent missing,
#       and codebook description.
# Why: with the scope filter applied, this table is short enough to scan end-to-end. The
#      pct_missing column surfaces sparse variables in the same view, so missingness no
#      longer needs its own section.

inventory = pd.DataFrame({
    'column': filtered_df.columns,
    'dtype': filtered_df.dtypes.astype(str).values,
    'non_null': filtered_df.notna().sum().values,
    'pct_missing': filtered_df.isna().mean().mul(100).round(1).values,
})
inventory = inventory.merge(codebook_df[['column', 'description']], on='column', how='left')
inventory

,column,dtype,non_null,pct_missing,description
0,country,object,5484,0.00,Geographic location.
1,year,int64,5484,0.00,Year of observation.
2,iso_code,object,5459,0.50,ISO 3166-1 alpha-3 three-letter country codes.
3,continent,object,5484,0.00,NaN
4,renewables_share_elec,float64,5296,3.40,Measured as a percentage of total electricity ...
5,renewables_share_energy,float64,2000,63.50,Measured as a percentage of the total primary ...
6,fossil_share_elec,float64,5296,3.40,"Electricity generation from coal, oil, and gas..."
7,fossil_share_energy,float64,2000,63.50,Measured as a percentage of the total primary ...
8,low_carbon_share_elec,float64,5296,3.40,Measured as a percentage of total electricity ...
9,low_carbon_share_energy,float64,2000,63.50,Measured as a percentage of the total primary ...



**Missing Values Assesment:**
- The `*_share_energy` columns are uniformly ~64% null. This is concentrated in smaller countries that don't report sector-level energy data (transport, heat, industry).
- The `*_share_elec` columns are mostly populated (~3–7% null) — electricity reporting is near-universal across countries with grids.
- The missingness is **systematic**, not random — it tracks country reporting capability, not data corruption.

## 5. Validate Key Metrics

- Check `renewables_share_energy + fossil_share_energy ≈ 100`
- Confirm partition logic is clean before building "% renewable" charts

In [13]:
# What: sum the renewable and fossil share columns row-wise and inspect the distribution.
# Why: if these two don't land near 100 for most rows, OWID's definitions don't partition
#      the way you'd assume — or there's a third category with missing data. Either way,
#      any "% renewable" chart built on them would mislead. Catching that now is cheaper
#      than rebuilding a Tableau view later.

mix_total = filtered_df[['renewables_share_energy', 'fossil_share_energy']].sum(axis=1, min_count=2)
mix_total.describe().round(2)

count   2000.00
mean      95.84
std        7.77
min       58.34
25%       94.48
50%      100.00
75%      100.00
max      100.00
dtype: float64

**Dashboard Note:** OWID's `renewables_share_energy` and `fossil_share_energy` together cover the primary energy mix and are the clean choice for any "% renewable" framing in the dashboard. If their pairwise sum drifts far from 100, revisit OWID's methodology before building stacked-area or share-comparison views.

## 6. Final Selection and Export

- Build country allow-list from full-coverage check (5 key share columns × 25 years)
- Write `data/processed/owid_energy_dashboard_subset.csv`

In [14]:
# Why: small-multiples view requires every country to have all 5 key share columns
#      non-null in every year. Filter to those countries, then write the CSV.

key_cols = [
    'renewables_share_energy', 'fossil_share_energy',
    'solar_share_energy', 'wind_share_energy', 'hydro_share_energy',
]
expected_years = filtered_df['year'].nunique()   # 25

coverage = (
    filtered_df.dropna(subset=key_cols, how='any')
    .groupby('country')['year'].nunique()
    .sort_values()
)

KEEP_COUNTRIES = coverage[coverage == expected_years].index.tolist()
print(f"Countries with full coverage: {len(KEEP_COUNTRIES)} of {filtered_df['country'].nunique()}")
print(f"\nFirst 20 (alphabetical): {sorted(KEEP_COUNTRIES)[:20]}")

OUTPUT_FILE = DATA_DIR / "processed" / "owid_energy_dashboard_subset_with_continent.csv"
subset_df = filtered_df[filtered_df['country'].isin(KEEP_COUNTRIES)].copy()
subset_df.to_csv(OUTPUT_FILE, index=False)
print(f"\nSubset exported: {OUTPUT_FILE.name}")
print(f"  Shape: {subset_df.shape[0]:,} rows × {subset_df.shape[1]} columns")

Countries with full coverage: 80 of 221

First 20 (alphabetical): ['Algeria', 'Argentina', 'Australia', 'Austria', 'Azerbaijan', 'Bangladesh', 'Belarus', 'Belgium', 'Brazil', 'Bulgaria', 'Canada', 'Chile', 'China', 'Colombia', 'Croatia', 'Cyprus', 'Czechia', 'Denmark', 'Ecuador', 'Egypt']



Subset exported: owid_energy_dashboard_subset_with_continent.csv
  Shape: 2,000 rows × 32 columns


**Dashboard Note:** The small-multiples view in the Tableau dashboard requires every panel (country) to have an unbroken series — gaps in any country's trendline render visibly and break the visual comparison. The "full coverage" filter applied above defines which countries make it into the dashboard subset.

---
## 7. Conclusion

**EDA Results:**

The OWID energy panel is curated and clean — no missing primary keys (apart from the intentionally-included `World` aggregate), no duplicates, dtypes already correct. The load-bearing work was scoping (130 → 32 columns, 1900–2025 → 2000–2024 years) and applying a strict full-coverage filter for the small-multiples view in Tableau (220 reporting countries → 79 with complete 25-year series across all 5 anchor share metrics, plus the `World` aggregate as a global benchmark).

**Key Findings:**
- **Scope:** Filtered to 5,484 country-year rows × 32 dashboard-relevant columns (down from 23,377 × 130, plus a derived `continent` column)
- **Primary Key:** All 220 country entities have non-null `iso_code` plus the `World` aggregate (1 row per year); zero duplicate `(country, year)` rows
- **Missing Data:** Systematic, not random — `*_share_energy` columns ~64% null (concentrated in smaller countries that don't report sector-level data); `*_share_elec` columns 3–7% null (electricity reporting is near-universal)
- **Metric Partition:** `renewables + fossil` energy shares partition cleanly (median sum = 100, mean = 95.84) — safe to use for "% renewable" framing
- **Final Selection:** 79 countries + `World` pass the strict 5-column × 25-year coverage filter, yielding a 2,000-row dashboard subset

**Limitations:**
- Smaller countries excluded by the strict coverage filter — dashboard reflects countries with mature energy reporting infrastructure, not the global universe
- 2024 reflects ~10% missing country reporting (199 of 220) — late-year values may be revised in subsequent OWID releases
- `gdp` has 30.7% missingness across the panel and should be used carefully if added to the dashboard later
- `World` is included as a benchmark series — must be filtered out of any chart that sums or ranks across countries to avoid double-counting

**Outputs:**
- `data/processed/owid_energy_dashboard_subset_with_continent.csv` — Dashboard-ready subset (79 countries + World × 25 years × 32 cols)

**Next Steps:**
Load `data/processed/owid_energy_dashboard_subset_with_continent.csv` into Tableau and build the dashboard views per `notes/tableau_course_project_spec.md`.